In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================
import numpy as np

import pandas as pd

import time

from scipy.stats import spearmanr

from sklearn.metrics import auc
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import onnxruntime as ort
import cv2
from sklearn.metrics import r2_score, auc
from numpy.linalg import norm
from scipy.stats import spearmanr
import time
from PIL import Image
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# Mutes every single UserWarning across all libraries
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# 2. CONFIGURATION
# ============================================================

DATA_PATH = "train_data.csv"

MOBILEVIT_ONNX_PATH = "mobilevit.onnx"
RESNET_ONNX_PATH = "Resnet.onnx"
RANDOM_FOREST_PATH = "best_random_forest_model.pkl"

IMG_SHAPE = (28, 28)
FEATURE_SIZE = 784

CLASS_NAMES = [
    "BENIGN",
    "DoS",
    "Fuzzy",
    "RPM",
    "Advanced",
    "Patator"
]

# ============================================================
# 3. CHECK FILES
# ============================================================

required_files = [
    DATA_PATH,
    MOBILEVIT_ONNX_PATH,
    RESNET_ONNX_PATH,
    RANDOM_FOREST_PATH
]

for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Missing file: {file}")

print("All required files found.")

# ============================================================
# 4. LOAD DATASET
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH, low_memory=False)

df.columns = df.columns.str.strip()

# Remove NaN and infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

df.reset_index(drop=True, inplace=True)

print(f"Dataset shape: {df.shape}")

# ============================================================
# 5. LABEL MAPPING
# ============================================================

def map_label(label):
    if label == "BENIGN":
        return "BENIGN"
    elif "DoS" in label:
        return "DoS"
    elif "PortScan" in label or "Brute Force" in label:
        return "Fuzzy"
    elif "Web Attack" in label or "Sql Injection" in label or "XSS" in label:
        return "RPM"
    elif "Bot" in label or "Infiltration" in label or "Heartbleed" in label:
        return "Advanced"
    elif "Patator" in label:
        return "Patator"
    else:
        return "Other"

df["Category"] = df["Label"].apply(map_label)

# Remove unwanted class
df = df[df["Category"] != "Other"]

print("\nClass Distribution:")
print(df["Category"].value_counts())

# ============================================================
# 6. FEATURE EXTRACTION
# ============================================================

print("\nPreparing features...")

features = df.select_dtypes(include=[np.number])
labels = df["Category"]

# ============================================================
# 7. FEATURE SCALING & SMART SUBSAMPLING
# ============================================================

scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# Convert to image pixel range
features_scaled = (features_scaled * 255).astype(np.uint8)

print(f"Total scaled features shape: {features_scaled.shape}")

# --- SMART STRATIFIED SUBSAMPLING TO EXACTLY 3000 SAMPLES ---
TARGET_SAMPLES = 3000
MIN_PER_CLASS = 15  # Enforce a minimum floor so downstream splits have enough samples

print(f"\nSubsampling dataset to {TARGET_SAMPLES} rows (with a floor of {MIN_PER_CLASS} per class)...")

sub_indices = []

# Step 1: Ensure every single class gets its minimum floor first
for category in df["Category"].unique():
    category_indices = df[df["Category"] == category].index.tolist()
    n_take = min(len(category_indices), MIN_PER_CLASS)
    
    # Safely sample the floor for this class
    chosen = pd.Series(category_indices).sample(n=n_take, random_state=42).tolist()
    sub_indices.extend(chosen)

# Step 2: Fill the remaining quota up to 3000 using the rest of the data
remaining_quota = TARGET_SAMPLES - len(sub_indices)

if remaining_quota > 0:
    remaining_indices = df.index.difference(sub_indices)
    remaining_df = df.loc[remaining_indices]
    
    # Exclude any categories that don't have enough remaining rows left to stratify
    rem_counts = remaining_df["Category"].value_counts()
    valid_categories = rem_counts[rem_counts >= 2].index
    strat_df = remaining_df[remaining_df["Category"].isin(valid_categories)]
    
    extra_indices, _ = train_test_split(
        strat_df.index.to_numpy(),
        train_size=remaining_quota,
        stratify=strat_df["Category"],
        random_state=42
    )
    sub_indices.extend(extra_indices)

# Step 3: Slice all your feature arrays and tracking DataFrames synchronously
features_scaled = features_scaled[sub_indices]
features = features.iloc[sub_indices].reset_index(drop=True)
labels = df["Category"].iloc[sub_indices].reset_index(drop=True)

print(f"Final features shape for pipeline: {features_scaled.shape}")
print("\nSubsampled Class Distribution (Safe for Splitting):")
print(labels.value_counts())

# ============================================================
# 8. CREATE IMAGE FUNCTION
# ============================================================

def create_image_from_row(row, size=FEATURE_SIZE):
    data = np.array(row)

    # Padding
    if len(data) < size:
        data = np.pad(
            data,
            (0, size - len(data)),
            mode='constant'
        )
    # Truncate if larger
    else:
        data = data[:size]

    img = data.reshape(IMG_SHAPE)
    return img.astype(np.uint8)

# ============================================================
# 9. IMAGE PREPROCESSING FOR ONNX
# ============================================================

def preprocess_resnet(img):
    img = cv2.resize(img, (224, 224))
    img = img.astype(np.float32) / 255.0
    img = np.stack([img] * 3, axis=-1)
    img = np.expand_dims(img, axis=0)
    return img.astype(np.float32)

def preprocess_mobilevit(img):
    img = img.astype(np.float32) / 255.0
    img = np.stack([img] * 3, axis=0)
    img = np.expand_dims(img, axis=0)
    return img.astype(np.float32)

# ============================================================
# 10. SOFTMAX FUNCTION
# ============================================================

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# ============================================================
# 11. LOAD ONNX MODELS
# ============================================================

print("\nLoading ONNX models...")

mobilevit_session = ort.InferenceSession(MOBILEVIT_ONNX_PATH)
resnet_session = ort.InferenceSession(RESNET_ONNX_PATH)

mobilevit_input_name = mobilevit_session.get_inputs()[0].name
resnet_input_name = resnet_session.get_inputs()[0].name

print("ONNX models loaded successfully.")

# ============================================================
# 12. GENERATE META FEATURES
# ============================================================

print("\nGenerating ensemble meta-features...")

meta_features = []
meta_labels = []

for idx in range(len(features_scaled)):
    row = features_scaled[idx]
    label = labels.iloc[idx]

    img = create_image_from_row(row)

    resnet_input = preprocess_resnet(img)
    mobilevit_input = preprocess_mobilevit(img)

    resnet_output = resnet_session.run(None, {resnet_input_name: resnet_input})[0]
    resnet_probs = softmax(resnet_output)

    mobilevit_output = mobilevit_session.run(None, {mobilevit_input_name: mobilevit_input})[0]
    mobilevit_probs = softmax(mobilevit_output)

    resnet_conf = np.max(resnet_probs[0])
    mobilevit_conf = np.max(mobilevit_probs[0])

    combined_features = np.concatenate([
        resnet_probs[0],
        mobilevit_probs[0],
        np.array([resnet_conf, mobilevit_conf])
    ])

    meta_features.append(combined_features)
    meta_labels.append(label)

    if idx % 1000 == 0:
        print(f"Processed {idx}/{len(features_scaled)} samples")

X_meta = np.array(meta_features)
y_meta = np.array(meta_labels)

print(f"\nMeta feature shape: {X_meta.shape}")

# ============================================================
# 13. LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_meta)

print("\nEncoded classes:")
print(label_encoder.classes_)

# ============================================================
# 14. TRAIN / TEST SPLIT (WITH INDEX TRACKING)
# ============================================================

indices = np.arange(len(X_meta))

train_idx, test_idx = train_test_split(
    indices, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

X_train, X_test = X_meta[train_idx], X_meta[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

X_test_original = features.iloc[test_idx].reset_index(drop=True)

print(f"\nTrain shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Original Test Features shape: {X_test_original.shape}")

# ============================================================
# 15. TRAIN FRESH RANDOM FOREST META-LEARNER
# ============================================================

from sklearn.ensemble import RandomForestClassifier

print("\nTraining a fresh Random Forest ensemble meta-learner...")
def compute_smoothed_weights(y, smooth_factor=0.15):
    # Count occurrences of each class
    classes, counts = np.unique(y, return_counts=True)
    total_samples = len(y)

    # Apply logarithmic smoothing
    # Ensure counts are not zero to avoid division by zero
    safe_counts = np.where(counts == 0, 1e-6, counts) # Use a small number instead of 0
    smoothed_weights_array = total_samples / (safe_counts ** smooth_factor)

    # Normalize so the minimum weight or average weight stays reasonable
    smoothed_weights_array /= np.mean(smoothed_weights_array)

    # Map class labels to their smoothed weights
    class_weight_dict = {cls: weight for cls, weight in zip(classes, smoothed_weights_array)}
    return class_weight_dict
class_weight_dict_selected = compute_smoothed_weights(y_train, smooth_factor=0.25)
# Initialize the Random Forest with standard, robust hyperparameters
rf_model = RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight=class_weight_dict_selected,
        n_estimators=300,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=3,
        max_features='log2')

# Train on the 14 meta-features generated from ONNX models
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

# Optional: Save the freshly trained model to disk for future use
fresh_rf_path = "fresh_random_forest_model.pkl"
joblib.dump(rf_model, fresh_rf_path)
print(f"Saved fresh model to: {fresh_rf_path}")

# ============================================================
# 16. EVALUATION (FIXED TO EVALUATE ON TEST SPLIT)
# ============================================================

print("\nEvaluating model on unseen test data...")

# FIX: Predict on X_test instead of X_train to check real performance
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {accuracy:.4f}")
print("\nTest Classification Report:\n")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

# ============================================================
# 17. RE-DEFINE FEATURE NAMES & WRAPPER FUNCTION
# ============================================================

feature_names = features.columns.tolist()

def end_to_end_predict_proba(X_tabular_numpy):
    """
    Black-box wrapper that converts raw tabular chunks from SHAP 
    back into images, runs ONNX models, and returns Random Forest probabilities.
    """
    X_scaled = scaler.transform(X_tabular_numpy)
    X_scaled = (X_scaled * 255).astype(np.uint8)
    
    meta_features_batch = []
    
    for row in X_scaled:
        img = create_image_from_row(row)
        
        resnet_input = preprocess_resnet(img)
        mobilevit_input = preprocess_mobilevit(img)
        
        res_out = resnet_session.run(None, {resnet_input_name: resnet_input})[0]
        res_probs = softmax(res_out)[0]
        
        mv_out = mobilevit_session.run(None, {mobilevit_input_name: mobilevit_input})[0]
        mv_probs = softmax(mv_out)[0]
        
        res_conf = np.max(res_probs)
        mv_conf = np.max(mv_probs)
        
        combined_features = np.concatenate([
            res_probs, 
            mv_probs, 
            np.array([res_conf, mv_conf])
        ])
        
        meta_features_batch.append(combined_features)
        
    return rf_model.predict_proba(np.array(meta_features_batch))

/home/minyar/onnx_xai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required files found.

Loading dataset...
Dataset shape: (1696725, 79)

Class Distribution:
Category
BENIGN      1362791
DoS          227842
Fuzzy         96186
Patator        8299
Advanced       1203
RPM             404
Name: count, dtype: int64

Preparing features...
Total scaled features shape: (1696725, 78)

Subsampling dataset to 3000 rows (with a floor of 15 per class)...
Final features shape for pipeline: (3000, 78)

Subsampled Class Distribution (Safe for Splitting):
Category
BENIGN      2352
DoS          406
Fuzzy        180
Patator       29
Advanced      17
RPM           16
Name: count, dtype: int64

Loading ONNX models...
ONNX models loaded successfully.

Generating ensemble meta-features...
Processed 0/3000 samples
Processed 1000/3000 samples
Processed 2000/3000 samples

Meta feature shape: (3000, 14)

Encoded classes:
['Advanced' 'BENIGN' 'DoS' 'Fuzzy' 'Patator' 'RPM']

Train shape: (2400, 14)
Test shape: (600, 14)
Original Test Features shape: (600, 78)

Training a fr

In [2]:
# ==========================================
# 18. PRE-COMPUTATION SETUP (Shared across tests)
# ==========================================
X_train_original = features.iloc[train_idx]
X_test_subsample = shap.sample(X_test_original, nsamples=5, random_state=42)
background_data = shap.sample(X_train_original, nsamples=5, random_state=42)
bg_vector = background_data.mean().values

num_classes = len(label_encoder.classes_)
num_samples = len(X_test_subsample)
num_features = X_test_subsample.shape[1]


# Extract Information Gain Weights
guidance_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
guidance_model.fit(X_train_original, labels.iloc[train_idx])
ig_scores = guidance_model.feature_importances_
epsilon = 1e-5
p_probs = (ig_scores + epsilon) / np.sum(ig_scores + epsilon)

# Correlation Matrix Mapping
corr_mat = X_train_original.corr().abs().fillna(0).values
tau = 0.9
C = {i: np.where((corr_mat[i] > tau) & (np.arange(num_features) != i))[0] for i in range(num_features)}

# ==========================================
# CONFIGURATION 4: Full Guided-Corr-Shap
# ==========================================
def explain_instance_corr_shap(X_target, K,tau):
    """
    Computes guided SHAP values. X_target can be a single sample or a batch.
    """
    # Convert DataFrame to numpy array if necessary
    if isinstance(X_target, pd.DataFrame):
        X_target_np = X_target.values
    elif len(np.shape(X_target)) == 1:
        X_target_np = X_target.reshape(1, -1)
    else:
        X_target_np = X_target
        
    n_samples = X_target_np.shape[0]
    shap_values = [np.zeros((n_samples, num_features)) for _ in range(num_classes)]

    for s_idx in range(n_samples):
        x = X_target_np[s_idx]
        X_with_all = []
        X_without_all = []
    
        for i in range(num_features):
            for _ in range(K):
                perm = np.random.choice(num_features, size=num_features, replace=False, p=p_probs)
                idx_i = np.where(perm == i)[0][0]
                S = perm[:idx_i]
            
                S_filtered = [j for j in S if j not in C[i]]
            
                x_with = bg_vector.copy()
                x_with[S_filtered] = x[S_filtered]
                x_with[i] = x[i]
            
                x_without = bg_vector.copy()
                x_without[S_filtered] = x[S_filtered]
            
                X_with_all.append(x_with)
                X_without_all.append(x_without)
            
        # WITH BATCHING: Matrix transformation execution
        X_with_all = np.array(X_with_all)
        X_without_all = np.array(X_without_all)
    
        preds_with = end_to_end_predict_proba(X_with_all)
        preds_without = end_to_end_predict_proba(X_without_all)
    
        marginal_all = preds_with - preds_without
        marginal_reshaped = marginal_all.reshape(num_features, K, num_classes)
        mean_marginal = np.mean(marginal_reshaped, axis=1)
    
        for c in range(num_classes):
            shap_values[c][s_idx, :] = mean_marginal[:, c]
            
    return shap_values

In [4]:
# ==========================================
# EVALUATION METRICS
# ==========================================

def calculate_local_fidelity(sample, K_budget,tau, class_index=0, num_perturbations=20):
    neighborhood = np.tile(sample, (num_perturbations, 1))
    noise = np.random.normal(0, 0.01, neighborhood.shape)
    perturbed_samples = neighborhood + noise
    
    # Get black-box target model predictions
    bb_preds = end_to_end_predict_proba(perturbed_samples)[:, class_index]
    
    # Generate custom explanations inline for our perturbation zone
    shap_vals_neighborhood = explain_instance_corr_shap(perturbed_samples, K_budget,tau)
    shap_vals_class = shap_vals_neighborhood[class_index]
    
    # Base value (prediction of the background vector)
    base_probs = end_to_end_predict_proba(bg_vector.reshape(1, -1))[0]
    
    # Reconstruct prediction sequence using additive SHAP formula
    shap_preds = base_probs[class_index] + shap_vals_class.sum(axis=1)
    
    return r2_score(bb_preds, shap_preds)

def calculate_robustness(sample, K_budget,tau, class_index=0, sigma=0.01):
    # Inject minor Gaussian noise to input feature space
    noise = np.random.normal(0, sigma, sample.shape)
    noisy_sample = sample + noise
    
    # Generate custom explanations for both baseline and perturbed inputs
    out_orig = explain_instance_corr_shap(sample, K_budget,tau)
    out_noisy = explain_instance_corr_shap(noisy_sample, K_budget,tau)
    
    # Extract attribution arrays
    shap_orig = out_orig[class_index][0]
    shap_noisy = out_noisy[class_index][0]
    
    # Calculate closeness of resulting attribution shifts using L2 Norm
    euclidean_dist = norm(shap_orig - shap_noisy)
    
    # Apply stability mapping scaling function
    return 1 / (1 + euclidean_dist)

def calculate_single_completeness_ratio(sample, shap_vals, k_percent=0.25):
    sample_arr = np.asarray(sample).flatten()
    bg_arr = np.asarray(bg_vector).flatten()
    shap_arr = np.asarray(shap_vals).flatten()
    
    # 1. Get original prediction probability vector for the true predicted class
    orig_prob_out = end_to_end_predict_proba(sample_arr.reshape(1, -1))[0]
    predicted_class = np.argmax(orig_prob_out)
    orig_confidence = orig_prob_out[predicted_class]
    
    if orig_confidence == 0:
        return 0.0
        
    # 2. Keep top K% features, replace the rest with background
    num_keep = max(1, int(len(shap_arr) * k_percent))
    top_indices = np.argsort(np.abs(shap_arr))[-num_keep:]
    
    masked_sample = bg_arr.copy()
    masked_sample[top_indices] = sample_arr[top_indices]
    
    masked_prob_out = end_to_end_predict_proba(masked_sample.reshape(1, -1))[0]
    masked_confidence = masked_prob_out[predicted_class]
    
    return masked_confidence / orig_confidence

In [ ]:

print("\n====================================================")
print("        ABLATION STUDY       ")
print("====================================================\n")

tau_values_to_test = [0.3, 0.5, 0.7, 0.9]
ablation_results = []

for current_t in tau_values_to_test:
    print(f"Evaluating tau = {current_t}...")
    current_k=15
    start_time = time.time()
    
    # 1. Base Explanation (Pass the dataset and K explicitly)
    run_A = explain_instance_corr_shap(X_test_subsample, K=15,tau=current_t)
    elapsed_time = time.time() - start_time
    
    run_A_flat = np.array(run_A).flatten()
    
    # 2. Compute Sparsity AUC
    abs_shap = np.abs(run_A_flat)
    if np.sum(abs_shap) == 0:
        sparsity_auc = 1.0
    else:
        sorted_shap = np.sort(abs_shap)[::-1]
        cumulative_importance = np.cumsum(sorted_shap) / np.sum(sorted_shap)
        x_axis = np.linspace(0, 1, len(cumulative_importance))
        sparsity_auc = auc(x_axis, cumulative_importance)
        
    # 3. Compute Stability (Spearman)
    run_B = explain_instance_corr_shap(X_test_subsample, K=15,tau=current_t)
    run_B_flat = np.array(run_B).flatten()
    corr_val, _ = spearmanr(run_A_flat, run_B_flat)
    stability_score = corr_val if not np.isnan(corr_val) else 1.0
    
    # 4. Compute New Local Metrics per sample
    fidelity_scores = []
    robustness_scores = []
    completeness_scores = []
    
    for s_idx in range(num_samples):
        sample_data = X_test_subsample.iloc[s_idx].values
        
        # Figure out the dominant predicted class to evaluate on
        pred_probs = end_to_end_predict_proba(sample_data.reshape(1, -1))[0]
        target_class = np.argmax(pred_probs)
        
        # Calculate Fidelity & Robustness
        fid = calculate_local_fidelity(sample_data, K_budget=current_k,tau=current_t, class_index=target_class)
        rob = calculate_robustness(sample_data, K_budget=current_k,tau=current_t, class_index=target_class)
        
        # Calculate Completeness using the pre-computed run_A SHAP values for the target class
        sample_shap = run_A[target_class][s_idx]
        comp = calculate_single_completeness_ratio(sample_data, sample_shap)
        
        fidelity_scores.append(fid)
        robustness_scores.append(rob)
        completeness_scores.append(comp)
    
    # 5. Log Results
    ablation_results.append({
        'tau': current_t,
        'Stability (Spearman)': stability_score,
        'Sparsity AUC': sparsity_auc,
        'Fidelity (R2)': np.mean(fidelity_scores),
        'Robustness': np.mean(robustness_scores),
        'Completeness Ratio': np.mean(completeness_scores),
        'Time per Run (sec)': elapsed_time
    })

# Format and display the results
df_ablation = pd.DataFrame(ablation_results)
df_ablation.set_index('tau', inplace=True)
df_ablation.to_csv('tau_ablation_study.csv')

In [6]:
df_ablation.set_index('tau', inplace=True)
df_ablation.to_csv('tau_ablation_study.csv')